# Language Modelling using N-gram

### **Task**
- Build and compare N-gram language models (unigram, bigram, trigram) using the Plain Text Wikipedia corpus to study generation quality and quantitative performance.

### **Objectives**
1. Load and inspect the dataset (Kaggle plain-text Wikipedia).
2. Preprocess text: lowercase, tokenize, and clean tokens.
3. Train unigram, bigram, and trigram models (estimate conditional probabilities).
4. Generate sample sentences from each model and compare qualitative fluency.
5. Evaluate models quantitatively using perplexity (with Add-1 smoothing) on a held-out test split.
6. Analyze limitations (data sparsity, model size, smoothing effects) and suggest improvements (better smoothing, backoff, pruning, or neural LMs).

### 1. Importing the modules and the dataset

In [14]:
import nltk
nltk.download('all') # Download all NLTK data packages (you can specify specific packages if you don't need everything)

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /home/rabi/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /home/rabi/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/rabi/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/rabi/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /home/rabi/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package ave

True

In [15]:
import pandas as pd

We used the Plain Text Wikipedia 2020-11 dataset as our text corpus for calculating our n-gram modelling.

In [16]:
import kagglehub
import os

# This downloads the dataset to your local cache
dataset_dir = kagglehub.dataset_download("ltcmdrdata/plain-text-wikipedia-202011")

# List all files inside the dataset
print("Files in dataset:")
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        # Print the relative paths you can use for dataset_load
        print(os.path.relpath(os.path.join(root, file), dataset_dir))

Files in dataset:
enwiki20201020/d9771113-ea97-46de-a812-27a5605ab884.json
enwiki20201020/6512b959-2b7b-42f7-a4e2-32734b33cd13.json
enwiki20201020/b2faad9e-dfa8-4107-bc35-0694725c10a5.json
enwiki20201020/f1dc01bf-7678-4453-b0ea-ad9ce9d5f084.json
enwiki20201020/301cdca4-a982-414d-b26d-1483028d7861.json
enwiki20201020/dd53bacd-ce88-4f03-9b06-ed75c9dec21e.json
enwiki20201020/98ba4472-9b7b-4168-b9e2-2b88393163da.json
enwiki20201020/14f66e46-f1fc-4d3d-8890-07d04d00e9dd.json
enwiki20201020/577897f1-71bf-45d2-a656-571504d8d40e.json
enwiki20201020/918bf1d8-97ca-4b91-9b64-b072775f960c.json
enwiki20201020/4e0d9991-5307-4c9b-bf98-979152fb67af.json
enwiki20201020/0b5e8bc0-af6c-48a3-8138-84557d78284b.json
enwiki20201020/b9826a38-4511-4ca7-b776-9e99e3cd9da1.json
enwiki20201020/2c3b989a-e17c-4041-97fe-ae3f3e1c4c9f.json
enwiki20201020/0d782c05-94c3-4454-9a6d-9978af11c419.json
enwiki20201020/998d802a-4092-45cc-ae9d-82691c9c7354.json
enwiki20201020/2ea5a70e-be96-485f-9c50-a6051cca4dde.json
enwiki2020102

In [17]:
import json
import pandas as pd
import os

# Construct the full file path to the JSON file you want to read
full_file_path = os.path.join(dataset_dir, "enwiki20201020/fbcef1c7-3f43-40e1-ba03-3b8f19ede846.json")

data = []
error_count = 0
limit = 10000  # How many rows you want to load to test

# line-by-line manual reading
with open(full_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# converting the clean data into a Pandas DataFrame
df = pd.DataFrame(data)

In [18]:
df.head()

,id,text,title
0,65475920,Berks/Dorset/Wilts 3 East was English Rugby Un...,Berks/Dorset/Wilts 3 East
1,65475934,Alfred Strasser (born 8 February 1954) is a re...,Alfred Strasser
2,65475936,Robert Milligan (born 1952) is a British rower...,Robert Milligan (rower)
3,65475938,Diversity debt is a term coined in the first h...,Diversity debt
4,65475940,Senesi is an Italian surname derived from the ...,Senesi


### 2. Preprocessing of text from the corpus

In [19]:
from collections import defaultdict, Counter
import random

# Download the NLTK punkt tokenizer if you haven't already
# nltk.download('punkt')

#1. Prepare the corpus by extracting text
sample_articles = df['text'].tolist()

# Combine them into one giant string
raw_corpus = " ".join(sample_articles)

#2. Tokenize the corpus
# Convert to lowercase and tokenize
tokens = nltk.word_tokenize(raw_corpus.lower())

# Optional: Filter out raw punctuation marks so the model generates cleaner sentences
tokens = [word for word in tokens if word.isalnum()]
print(f"Sample tokens: {tokens[:20]}")
print(f"Total tokens extracted: {len(tokens)}")

Sample tokens: ['3', 'east', 'was', 'english', 'rugby', 'union', 'league', 'forming', 'part', 'of', 'the', 'south', 'west', 'division', 'for', 'clubs', 'primarily', 'based', 'in', 'berkshire']
Total tokens extracted: 3425843


After preprocessing the tokens, we find that the number of tokens in the corpus we are working on is almost 3.4M. For now let us work on building the model.

### 3. Building the N-gram models

In [20]:
print("Building the Trigram Model...")
trigrams = list(nltk.trigrams(tokens))
trigram_model = defaultdict(Counter)

# Count frequencies
for *context, target in trigrams:
    trigram_model[tuple(context)][target] += 1

# Convert counts into probabilities
for context in trigram_model:
    total_count = float(sum(trigram_model[context].values()))
    for target in trigram_model[context]:
        trigram_model[context][target] /= total_count

print(f"Model trained! Unique context windows: {len(trigram_model)}")

Building the Trigram Model...
Model trained! Unique context windows: 1223382


In [21]:
print("Building the Bigram Model...")
bigrams = list(nltk.bigrams(tokens))
bigram_model = defaultdict(Counter)

# Count frequencies
for context, target in bigrams:
    bigram_model[context][target] += 1

# Convert counts into probabilities
for context in bigram_model:
    total_count = float(sum(bigram_model[context].values()))
    for target in bigram_model[context]:
        bigram_model[context][target] /= total_count

print(f"Bigram model trained! Unique context windows: {len(bigram_model)}")

Building the Bigram Model...
Bigram model trained! Unique context windows: 151773


In [22]:
print("Building the Unigram Model...")
unigrams = tokens
unigram_model = Counter(unigrams)

# Convert counts into probabilities
total_unigram_count = float(sum(unigram_model.values()))
for token in unigram_model:
    unigram_model[token] /= total_unigram_count

print(f"Unigram model trained! Unique tokens: {len(unigram_model)}")

Building the Unigram Model...
Unigram model trained! Unique tokens: 151773


### 4. Generating sentences from the model

In [23]:
def generate_trigram(trigram_model, seed=None, valid_contexts=None, length=20):
    # seed can be a 2-tuple context, a single token, or None
    if seed is None:
        seed = random.choice(valid_contexts)
    elif isinstance(seed, str):
        candidates = [c for c in valid_contexts if c[0] == seed]
        seed = random.choice(candidates) if candidates else random.choice(valid_contexts)
    context = tuple(seed)
    generated = list(context)
    for _ in range(length):
        if context not in trigram_model:
            break
        choices = list(trigram_model[context].keys())
        probs = list(trigram_model[context].values())
        next_word = random.choices(choices, weights=probs)[0]
        generated.append(next_word)
        context = tuple(generated[-2:])
    return " ".join(generated).capitalize() + "."


In [24]:
def generate_bigram(bigram_model, seed=None, length=20):
    if seed is None:
        seed = random.choice(list(bigram_model.keys()))
    current = seed
    generated = [current]
    for _ in range(length):
        if current not in bigram_model:
            break
        choices = list(bigram_model[current].keys())
        probs = list(bigram_model[current].values())
        next_word = random.choices(choices, weights=probs)[0]
        generated.append(next_word)
        current = next_word
    return " ".join(generated).capitalize() + "."

In [25]:
def generate_unigram(unigram_model, length=20):
    choices = list(unigram_model.keys())
    probs = list(unigram_model.values())
    generated = [random.choices(choices, weights=probs)[0] for _ in range(length)]
    return " ".join(generated).capitalize() + "."
# Examples


In [26]:
valid_contexts = list(trigram_model.keys()) # needed for trigram generation

# Generate and compare samples
print("\nUnigram sample:", generate_unigram(unigram_model, length=30))
print("Bigram sample:", generate_bigram(bigram_model, length=30))
print("Trigram sample:", generate_trigram(trigram_model, valid_contexts=valid_contexts, length=30))

# Comparison
print("\n=== Model Comparison ===")
print(f"Unigram - Unique tokens: {len(unigram_model)}")
print(f"Bigram - Unique contexts: {len(bigram_model)}")
print(f"Trigram - Unique contexts: {len(trigram_model)}")


Unigram sample: Kars he and respiratory poetry extreme a facebook 2010 democratic unchanged myanmar from dog the hojjatoleslam births rams for maynard personal richard category 800 assassins drained greater an to porter.
Bigram sample: Lindveit 1960 to the railway stations established that led by abd al 2011k zhang et voies de são paulo brazil and revocation of birth of british mps rebelled against fc baden.
Trigram sample: Europe recording victories in the outskirts of the population was forced to face with queer whiskers red hair and beard and a lintel with a square plan they are sometimes three to.

=== Model Comparison ===
Unigram - Unique tokens: 151773
Bigram - Unique contexts: 151773
Trigram - Unique contexts: 1223382


From the unique context counts we calculated just now, we can infer the following things about the three models:

1. **Unigram Model (N=1):**
 * It is a complete word salad, which selects words based purely on their overall frequency in the dataset, without any comprehension about the word order. The sentences are grammatically incorrect and completely nonsensical.

2. **Bigram Model (N=2):**
* It brings us into a fascinating paradox: the sentence which the model generated made perfect sense in the immediate context, but falls apart when the contexts are joined together. This is because the model only remembers the immediately preceding word.

3. **Trigram Model (N=3):**
* The model generates realistic-sounding text from the corpus, but often copies exact sentences from the training data because specific 3-word combinations are rare.

We also notice how the unique context counts drastically increase as $N$ increases:

* **Unigram**: Smallest memory footprint (only tracking individual words).
* **Bigram**: Tracks pairs of words (e.g., 151k+ contexts).
* **Trigram**: Tracks triplets, resulting in a massive footprint (e.g., 1.2M+ contexts).

This massive expansion is why larger N-gram models often hit "dead ends" on limited datasets and require smoothing (like Laplace) or Backoff strategies to survive unseen sequences!

### 5. Evaluation of the model

In [27]:
import math
from collections import Counter
from nltk.util import ngrams

# 1. Split your tokens into Train (80%) and Test (20%)
split_index = int(len(tokens) * 0.8)
train_tokens = tokens[:split_index]
test_tokens = tokens[split_index:]

# We need the vocabulary size for Laplace Smoothing
vocab_size = len(set(train_tokens))
print(f"Training tokens: {len(train_tokens)} | Testing tokens: {len(test_tokens)}")
print(f"Vocabulary Size: {vocab_size}\n")

def calculate_perplexity(n, train_set, test_set, vocab_size):
    """Trains an N-gram model and calculates perplexity on a test set using Add-1 Smoothing."""
    context_counts = Counter()
    ngram_counts = Counter()
    
    # Train the model
    for ngram in ngrams(train_set, n):
        context = ngram[:-1] # Everything except the last word
        context_counts[context] += 1
        ngram_counts[ngram] += 1
        
    # Test the model
    log_prob_sum = 0
    N_test = len(test_set) - n + 1
    
    for ngram in ngrams(test_set, n):
        context = ngram[:-1]
        
        # Add-1 Smoothing Formula
        # P = (count(ngram) + 1) / (count(context) + vocab_size)
        prob = (ngram_counts[ngram] + 1) / (context_counts[context] + vocab_size)
        log_prob_sum += math.log(prob)
        
    # Calculate final perplexity
    perplexity = math.exp(-log_prob_sum / N_test)
    return perplexity

# 2. Run the comparison
print("Calculating Perplexities (Lower is better)...")

# Unigram (n=1)
# Note: For unigrams, the context is empty, so context count is just the total number of words.
unigram_ppl = calculate_perplexity(1, train_tokens, test_tokens, vocab_size)
print(f"Unigram Perplexity: {unigram_ppl:.2f}")

# Bigram (n=2)
bigram_ppl = calculate_perplexity(2, train_tokens, test_tokens, vocab_size)
print(f"Bigram Perplexity:  {bigram_ppl:.2f}")

# Trigram (n=3)
trigram_ppl = calculate_perplexity(3, train_tokens, test_tokens, vocab_size)
print(f"Trigram Perplexity: {trigram_ppl:.2f}")

Training tokens: 2740674 | Testing tokens: 685169
Vocabulary Size: 130068

Calculating Perplexities (Lower is better)...
Unigram Perplexity: 3539.57
Bigram Perplexity:  15230.90
Trigram Perplexity: 65176.52


Although higher-order N-gram models usually achieve lower perplexity, our results show the exact opposite: perplexity skyrockets from Unigram (3,539) to Trigram (65,176). This counterintuitive spike is caused by severe data sparsity combined with Laplace (Add-1) smoothing. With a massive vocabulary of over 130,000 words but only 2.7 million training tokens, the model encountered countless unseen Bigrams and Trigrams during testing. To prevent zero-probability errors, Laplace smoothing essentially "steals" probability mass from known sequences and distributes it evenly across billions of possible but unseen word combinations, mathematically crippling the Bigram and Trigram models on such a limited dataset.